# 4. Monitoreo de modelos en produccion

Simulamos el monitoreo post-despliegue comparando la ventana de
**referencia** (`step <= 400`, usada para entrenar) contra la ventana de
**produccion** (`step > 400`), tal como lo exige el proyecto. Cubrimos
(Sesion 7 y 9-10 del curso):

- **Data drift**: ¿cambio la distribucion de las features de entrada?
  (KS test, PSI, reporte Evidently).
- **Target drift / concept drift**: ¿cambio la distribucion del target o
  la relacion features→target?
- **Model (performance) drift**: ¿se degrado el desempeno del modelo en
  produccion respecto al offline, y a lo largo del tiempo (por bloques de
  `step`)?

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
# Nota: en evidently >= 0.6 la API "clasica" (Report + metric_preset) vive
# bajo el namespace evidently.legacy; evidently.legacy.* es el shim oficial
# de compatibilidad hacia atras, mismo API que se vio en el curso (Sesion 9-10).
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import (
    ClassificationPreset,
    DataDriftPreset,
    TargetDriftPreset,
)

from src.config import PROCESSED_DIR, REFERENCE_DATA_PATH, REPORTS_DIR, TARGET_COL
from src.features import select_model_features

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

reference = pd.read_csv(REFERENCE_DATA_PATH)
production = pd.read_csv(PROCESSED_DIR / "production_scored.csv")

FEATURES = select_model_features(reference)

# Para DATA DRIFT y TARGET DRIFT usamos el reference.csv completo (train+test):
# no involucra predicciones del modelo, asi que no hay riesgo de fuga -- queremos
# la distribucion completa de referencia, no solo un subconjunto held-out.
reference = reference.rename(columns={TARGET_COL: "target"})
production = production.rename(columns={TARGET_COL: "target", "fraud_pred": "prediction"})

cols = FEATURES + ["target"]
ref_view = reference[cols]
prod_view = production[cols + ["prediction"]] if "prediction" in production else production[cols]

# Para PERFORMANCE DRIFT (ClassificationPreset) SI importa que las predicciones
# de referencia sean honestas: usamos el holdout X_test ya scoreado en el
# notebook de entrenamiento (nunca visto por el modelo), no el reference.csv
# completo -- scorear el champion sobre filas con las que fue entrenado seria
# fuga de informacion (prediccion in-sample, artificialmente optimista).
ref_test_scored_path = PROCESSED_DIR / "reference_test_scored.csv"
if ref_test_scored_path.exists():
    reference_test_scored = pd.read_csv(ref_test_scored_path)
    reference_test_scored = reference_test_scored.rename(
        columns={TARGET_COL: "target", "fraud_pred": "prediction"}
    )
    ref_perf_view = reference_test_scored[cols + ["prediction"]]
else:
    ref_perf_view = None
    print(
        "Aviso: no se encontro reference_test_scored.csv -- corre de nuevo "
        "2.0-entrenamiento-modelo.ipynb para generarlo."
    )


## 4.1 Data drift (features de entrada)

In [ ]:
data_drift_report = Report(metrics=[DataDriftPreset()])
data_drift_report.run(reference_data=ref_view[FEATURES], current_data=prod_view[FEATURES])
data_drift_report.save_html(str(REPORTS_DIR / "data_drift_report.html"))
data_drift_report

## 4.2 Target drift

In [ ]:
target_drift_report = Report(metrics=[TargetDriftPreset()])
target_drift_report.run(
    reference_data=ref_view[FEATURES + ["target"]],
    current_data=prod_view[FEATURES + ["target"]],
)
target_drift_report.save_html(str(REPORTS_DIR / "target_drift_report.html"))
target_drift_report

## 4.3 Model / performance drift

Comparamos el desempeno del modelo sobre el **holdout de referencia**
(`X_test`, nunca visto durante el entrenamiento) contra **produccion**, con
el mismo `ClassificationPreset`. Usamos a proposito el holdout y no todo
`reference.csv`: scorear el modelo sobre filas con las que fue entrenado
seria fuga de informacion (prediccion in-sample, artificialmente optimista).


In [ ]:
if "prediction" in prod_view and ref_perf_view is not None:
    perf_report = Report(metrics=[ClassificationPreset()])
    perf_report.run(
        reference_data=ref_perf_view,
        current_data=prod_view[FEATURES + ["target", "prediction"]],
    )
    perf_report.save_html(str(REPORTS_DIR / "classification_performance_report.html"))
    perf_report
else:
    print("Corre antes 2.0-entrenamiento-modelo.ipynb para generar los CSV scoreados")

## 4.4 Desempeno por bloques de tiempo (concept/model drift a lo largo de `step`)

In [ ]:
import numpy as np
from src.metrics import classification_metrics

if "prediction" in production.columns:
    prod_blocks = production.copy()
    prod_blocks["step_block"] = pd.cut(prod_blocks["step"], bins=10)

    rows = []
    for block, grp in prod_blocks.groupby("step_block", observed=True):
        if grp["target"].sum() == 0:
            continue
        m = classification_metrics(grp["target"], grp["prediction"], grp["fraud_proba"])
        m["step_block"] = str(block)
        m["n_transacciones"] = len(grp)
        m["n_fraudes"] = int(grp["target"].sum())
        rows.append(m)

    drift_over_time = pd.DataFrame(rows)
    drift_over_time.to_csv(PROCESSED_DIR / "performance_drift_over_time.csv", index=False)
    drift_over_time

In [ ]:
import matplotlib.pyplot as plt

if "prediction" in production.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(drift_over_time["step_block"].astype(str), drift_over_time["pr_auc"], marker="o", label="PR-AUC")
    ax.plot(drift_over_time["step_block"].astype(str), drift_over_time["recall"], marker="o", label="Recall")
    ax.set_xticklabels(drift_over_time["step_block"].astype(str), rotation=45, ha="right")
    ax.set_title("Desempeno del modelo por bloque de tiempo (produccion)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORTS_DIR.parent / "reports" / "performance_drift_over_time.png", bbox_inches="tight")
    plt.show()

## 4.5 Conclusiones de monitoreo

_(Completar tras correr con datos reales: ¿hay drift significativo en
alguna feature clave (`amount`, `errorBalanceOrig`, etc.)? ¿Se mantiene el
PR-AUC/recall estable a lo largo de los bloques de `step`, o se degrada?
¿Que estrategia de las vistas en Sesion 10 (no hacer nada / retraining /
champion-challenger / ensemble) se recomienda?)_

Los reportes HTML generados aqui (`app/reports/*.html`) alimentan la
**app de monitoreo** en [`app/monitoring_app.py`](../app/monitoring_app.py).